In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np


In [2]:
con = sqlite3.connect("vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "v33.db" AS v33')

## 1. tabel 
## verb -> palju esineb obl+kääne (6 kohakäänet) : mitu matchi ja mitu distinct root 

Võtta välja kõik mis on kohakäändes ja sõna deprel on obl

#### alustabel, kus on ainult obl ja kohakäänetes verbid

In [73]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes as
SELECT distinct
    tr.head_id as head_id,
    tbl1.verb as verb,
    tr.id as transaction_id,
    tr.lemma as root_word,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus

FROM 

transaction_head as tbl1

join 

transaction_v2 as tr

on 
    tbl1.id = tr.head_id
    
where
tr.deprel = 'obl'
and 
(INSTR(',' || tr.feats || ',', ',' || 'abl' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'adit' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'all' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ad' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'el' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'ill' || ',') > 0
or INSTR(',' || tr.feats || ',', ',' || 'in' || ',') > 0
)
""")



CPU times: user 18 s, sys: 4.13 s, total: 22.1 s
Wall time: 37.7 s


#### alustabelisse juurde veergu 'kaane', mis käändega on tegu

In [74]:
cur.execute("""
ALTER TABLE transactions_verbs_obl_kohakaandes
ADD kaane VARCHAR(50)
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'abl'
WHERE INSTR(',' || tr_feats || ',', ',' || 'abl' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'adit'
WHERE INSTR(',' || tr_feats || ',', ',' || 'adit' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'all'
WHERE INSTR(',' || tr_feats || ',', ',' || 'all' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'ad'
WHERE INSTR(',' || tr_feats || ',', ',' || 'ad' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'el'
WHERE INSTR(',' || tr_feats || ',', ',' || 'el' || ',') > 0
""")
con.commit()


cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'ill'
WHERE INSTR(',' || tr_feats || ',', ',' || 'ill' || ',') > 0
""")
con.commit()

cur.execute("""
UPDATE transactions_verbs_obl_kohakaandes
SET kaane = 'in'
WHERE INSTR(',' || tr_feats || ',', ',' || 'in' || ',') > 0
""")
con.commit()


#### vaade mis on tabelis

In [31]:
tables = cur.execute("""
SELECT * FROM transactions_verbs_obl_kohakaandes
limit 10
""")

for i, elem in enumerate(tables):
    print(elem)

(2, 'toimuma', 1, 'lõpp', 'obl', 'S', 'com,in,sg', 'UNK', 'UNK', 'in')
(3, 'saama', 7, 'keel', 'obl', 'S', 'all,com,pl', 'UNK', 'UNK', 'all')
(10, 'tulema', 19, 'sina', 'obl', 'P', 'ad,sg', 'UNK', 'YES', 'ad')
(11, 'viilima', 22, 'tund', 'obl', 'S', 'com,el,pl', 'UNK', 'UNK', 'el')
(11, 'viilima', 23, 'juht', 'obl', 'S', 'ad,com,sg', 'UNK', 'YES', 'ad')
(25, 'muutuma', 40, 'mis', 'obl', 'P', 'el,sg', 'UNK', 'UNK', 'el')
(33, 'minema', 59, 'rahvas', 'obl', 'S', 'all,com,sg', 'UNK', 'UNK', 'all')
(37, 'tekkima', 69, 'see', 'obl', 'P', 'el,sg', 'UNK', 'UNK', 'el')
(51, 'kutsuma', 85, 'elu', 'obl', 'S', 'adit,com,sg', 'UNK', 'UNK', 'adit')
(53, 'tulema', 88, 'toim', 'obl', 'S', 'adit,com,sg', 'UNK', 'UNK', 'adit')


### base tabel, kus on distinct verbid eelmisest tabelist ja iga kohakäände jaoks count veerg algväärtusega 0

In [3]:
%%time

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_base
""")

cur.execute("""
Create table transactions_verbs_obl_kohakaandes_root_counts_base as
SELECT distinct verb, 
0 as abl_cnt, 
0 as adit_cnt, 
0 as all_cnt, 
0 as ad_cnt, 
0 as el_cnt, 
0 as ill_cnt, 
0 as in_cnt

FROM 

transactions_verbs_obl_kohakaandes

""")


CPU times: user 2.05 s, sys: 111 ms, total: 2.16 s
Wall time: 2.21 s


### Täida tabel distinct root countidega

In [4]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_root_counts_base
"""

source = pd.read_sql_query(query, con)

In [5]:
for i in tqdm(range(len(source))):
    verb = source.iloc[i]["verb"]
    #print(verb)
    
    for case in ["abl", "adit", "all", "ad", "el", "ill", "in"]:
        count = 0
        res = cur.execute("""
                select count(distinct root_word) from transactions_verbs_obl_kohakaandes
                where verb='{v}'
                and INSTR(',' || tr_feats || ',', ',' || '{c}' || ',') > 0
                """.format(v=verb, c=case))
        for e in res:
            count = e[0]
        #print(count)
        source.at[i, case+'_cnt'] = count
    #break

100%|█████████████████████████████████████| 9618/9618 [6:42:48<00:00,  2.51s/it]


In [6]:
source

,verb,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt
0,toimuma,217,314,1026,2953,1076,164,6747
1,saama,5841,1510,6175,5967,21132,1263,9107
2,tulema,3199,2242,7121,9166,9139,2353,6342
3,viilima,2,0,3,16,29,1,14
4,muutuma,166,152,1180,1450,1288,121,2097
...,...,...,...,...,...,...,...,...
9613,lastnuma,0,0,0,0,0,0,1
9614,naajuma,0,0,0,0,0,0,1
9615,sekskima,0,0,0,1,0,0,0
9616,ampima,0,0,0,0,1,0,0


In [13]:
# salvesta

cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_distinct
""")

source.to_sql(name='transactions_verbs_obl_kohakaandes_root_counts_distinct', con=con)

source.to_csv("transactions_verbs_obl_kohakaandes_root_counts_distinct.csv", index=False, encoding="utf-8", sep=",")

### Täida tabel root matchide countidega

In [32]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_root_counts_base
"""

source = pd.read_sql_query(query, con)

In [21]:
for i in tqdm(range(len(source))):
    verb = source.iloc[i]["verb"]
    #print(verb)
    
    for case in ["abl", "adit", "all", "ad", "el", "ill", "in"]:
        count = 0
        res = cur.execute("""
                select count(root_word) from transactions_verbs_obl_kohakaandes
                where verb='{v}'
                and INSTR(',' || tr_feats || ',', ',' || '{c}' || ',') > 0
                """.format(v=verb, c=case))
        for e in res:
            count = e[0]
        #print(count)
        source.at[i, case+'_cnt'] = count
    #break

100%|█████████████████████████████████████| 9618/9618 [6:18:48<00:00,  2.36s/it]


In [22]:
source

,verb,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt
0,toimuma,337,676,2127,51404,3077,274,50217
1,saama,25657,24535,33372,83941,111688,4723,64109
2,tulema,11916,34603,55260,104553,44188,15199,36534
3,viilima,3,0,3,19,37,1,15
4,muutuma,304,201,2991,8985,3580,168,7416
...,...,...,...,...,...,...,...,...
9613,lastnuma,0,0,0,0,0,0,1
9614,naajuma,0,0,0,0,0,0,1
9615,sekskima,0,0,0,1,0,0,0
9616,ampima,0,0,0,0,1,0,0


In [23]:
cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_root_counts_matches
""")

source.to_sql(name='transactions_verbs_obl_kohakaandes_root_counts_matches', con=con)

source.to_csv("transactions_verbs_obl_kohakaandes_root_counts_matches.csv", index=False, encoding="utf-8", sep=",")

## 2. tabel

### iga verb+obl+kohakääne jaoks count elus ja count koht, count kokku

1) count distinct root

2) count matches

## base tabel kus on verb, kääne, elus_cnt, koht_cnt, distinct root count  

In [3]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base AS
select distinct verb,kaane, verb||'_'||kaane as verb_kaane,
0 as elus_cnt,
0 as koht_cnt,
count(distinct root_word) as root_cnt
from transactions_verbs_obl_kohakaandes
group by verb,kaane
order by root_cnt desc
--order by verb
""")

CPU times: user 6.37 s, sys: 318 ms, total: 6.69 s
Wall time: 6.7 s


In [7]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct_base
"""

source = pd.read_sql_query(query, con)
source

,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,saama,el,saama_el,0,0,21132
1,andma,all,andma_all,0,0,12526
2,rääkima,el,rääkima_el,0,0,10930
3,jääma,el,jääma_el,0,0,9936
4,tulema,ad,tulema_ad,0,0,9166
...,...,...,...,...,...,...
30081,šveitsima,el,šveitsima_el,0,0,1
30082,švipsima,ad,švipsima_ad,0,0,1
30083,žestikuleerima,ad,žestikuleerima_ad,0,0,1
30084,žisraelima,ad,žisraelima_ad,0,0,1


In [8]:
for i in tqdm(range(len(source))):
    verb = source.iloc[i]["verb"]
    kaane = source.iloc[i]["kaane"]
    #print(verb)
    
    for case in ["elus", "koht"]:
        count = 0
        res = cur.execute("""
                select count(distinct root_word) from transactions_verbs_obl_kohakaandes
                where verb='{v}'
                and kaane = '{k}'
                and {c}='YES'
                """.format(v=verb, k=kaane, c=case))
        for e in res:
            count = e[0]
        #print(count)
        source.at[i, case+'_cnt'] = count
    #break

100%|███████████████████████████████████| 30086/30086 [6:08:20<00:00,  1.36it/s]


In [9]:
source

,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,saama,el,saama_el,1317,408,21132
1,andma,all,andma_all,1649,267,12526
2,rääkima,el,rääkima_el,805,202,10930
3,jääma,el,jääma_el,709,265,9936
4,tulema,ad,tulema_ad,1178,174,9166
...,...,...,...,...,...,...
30081,šveitsima,el,šveitsima_el,0,0,1
30082,švipsima,ad,švipsima_ad,0,0,1
30083,žestikuleerima,ad,žestikuleerima_ad,0,1,1
30084,žisraelima,ad,žisraelima_ad,0,0,1


In [10]:
cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct
""")

source.to_sql(name='transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct', con=con)

## base tabel kus on verb, kääne, elus_cnt, koht_cnt, root count  

In [3]:
%%time

cur.execute("""
DROP table if exists transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches_base

""")

cur.execute("""
CREATE TABLE transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches_base AS
select distinct verb,kaane, verb||'_'||kaane as verb_kaane,
0 as elus_cnt,
0 as koht_cnt,
count(root_word) as root_cnt
from transactions_verbs_obl_kohakaandes
group by verb,kaane
order by root_cnt desc
--order by verb
""")

CPU times: user 4.69 s, sys: 456 ms, total: 5.14 s
Wall time: 8.82 s


In [33]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches_base
"""

source = pd.read_sql_query(query, con)


In [8]:
for i in tqdm(range(len(source))):
    verb = source.iloc[i]["verb"]
    kaane = source.iloc[i]["kaane"]
    #print(verb)
    
    for case in ["elus", "koht"]:
        count = 0
        res = cur.execute("""
                select count(root_word) from transactions_verbs_obl_kohakaandes
                where verb='{v}'
                and kaane = '{k}'
                and {c}='YES'
                """.format(v=verb, k=kaane, c=case))
        for e in res:
            count = e[0]
        #print(count)
        source.at[i, case+'_cnt'] = count
    #break

100%|███████████████████████████████████| 30086/30086 [6:09:06<00:00,  1.36it/s]


In [9]:
source

,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,saama,el,saama_el,13448,5330,111688
1,tulema,ad,tulema_ad,24980,3147,104553
2,andma,all,andma_all,34988,3385,89057
3,saama,ad,saama_ad,4303,2893,83941
4,olema,ad,olema_ad,22693,4253,75894
...,...,...,...,...,...,...
30081,šveitsima,el,šveitsima_el,0,0,1
30082,švipsima,ad,švipsima_ad,0,0,1
30083,žestikuleerima,ad,žestikuleerima_ad,0,1,1
30084,žisraelima,ad,žisraelima_ad,0,0,1


In [10]:
cur.execute("""
DROP TABLE IF EXISTS transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches
""")

source.to_sql(name='transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches', con=con)

In [183]:
con.close()